# Tutorial HMSpectralGun

Notebook pratico per imparare a usare HMSpectralGun in tre modalita:
- sintesi seriale (`main.py`)
- sintesi parallela (`main_parallel.py`)
- analisi line-by-line (`line_chi2_on_the_fly.py`)


## 1. Prerequisiti

Assicurati di avere installato le dipendenze Python principali: `numpy`, `pandas`, `scipy`, `tqdm`, `mendeleev`, `matplotlib`, `PyAstronomy`.

Nota: alcuni path nel codice sono hardcoded (dataset MARCS, cartella COM, eseguibili Turbospectrum). Se sei su un altro sistema, prima aggiorna quei path.

In [ ]:
from pathlib import Path
import json
import textwrap

repo = Path.cwd()
repo

## 2. Crea una sandbox tutorial

Questa sezione prepara file di esempio senza toccare i tuoi input di produzione.

In [ ]:
tutorial_dir = repo / 'tutorial_data'
out_dir = tutorial_dir / 'output_spectra'
model_dir = tutorial_dir / 'models'
linelist_dir = tutorial_dir / 'linelists'
obs_dir = tutorial_dir / 'obs'
results_dir = tutorial_dir / 'results'
synth_dir = tutorial_dir / 'synthetic_line_fit'

for d in [tutorial_dir, out_dir, model_dir, linelist_dir, obs_dir, results_dir, synth_dir]:
    d.mkdir(parents=True, exist_ok=True)

tutorial_dir

## 3. Genera template file (`input.ts`, `abu`, `linelist`, `line file`, `config JSON`)

Questi file sono un punto di partenza: adatta i nomi dei modelli, le righe spettrali e i path reali.

In [ ]:
input_ts = tutorial_dir / 'input_tutorial.ts'
abu_file = tutorial_dir / 'abu_tutorial.ts'
linelist_index = tutorial_dir / 'linelist_tutorial.ts'
line_file = tutorial_dir / 'lines_tutorial'
obs_file = obs_dir / 'star_demo.spec'
cfg_file = tutorial_dir / 'line_fit_config.tutorial.json'

input_ts.write_text(textwrap.dedent(f'''\
{out_dir}/
{linelist_dir}/
{model_dir}/
ExplicitModel=False
interp=nearest
NLTE=False
3600,1.0  -1.00  0.20  15000  15100  2.0  st  *  28000  0.02  *  {linelist_index.name}  {abu_file.name}  *  txt
'''))

abu_file.write_text(textwrap.dedent('''\
8 0.20
12 0.20
14 0.20
16 0.20
20 0.20
22 0.20
26 0.00
612613 15.0
'''))

linelist_index.write_text('dummy_linelist_file.lin\n')
(linelist_dir / 'dummy_linelist_file.lin').write_text('# metti qui una vera linelist Turbospectrum\n')

line_file.write_text(textwrap.dedent('''\
15020.000 Fe 1 2.18 -3.10
15035.500 Fe 1 3.20 -1.40
'''))

# osservato fittizio minimale (wavelength flux err telluric)
with obs_file.open('w') as f:
    for i in range(200):
        w = 15000.0 + i * 0.25
        f.write(f"{w:.3f} 1.000 0.010 0.000\n")

cfg = {
    'paths': {
        'obs_spectra_path': str(obs_dir),
        'analysis_synth_path': str(synth_dir),
        'results_path': str(results_dir),
    },
    'keywords': {
        'auto': True,
        'mcmc': False,
        'n_mcmc': 100,
        'synth_half_window': 3.0,
        'chi2_half_window': 0.35,
        'default_err': 0.01,
        'xfe_min': -0.5,
        'xfe_max': 0.5,
        'xfe_step': 0.1,
        'interp': 'nearest',
        'nlte': 'False',
        'chemistry': 'st',
        'resnum': 0.02,
        'extension': 'txt',
        'monoelem': '*',
        'linelist_library_path': str(linelist_dir),
        'aux_file_path': str(tutorial_dir),
        'keep_synthetic': True,
        'cleanup_com_files': True,
        'optimize_shift': True,
        'optimize_norm': True,
        'optimize_resolution': True
    },
    'stars': [
        {
            'name_obs_spec': obs_file.name,
            'teff': 3600,
            'logg': 1.0,
            '[Fe/H]': -1.0,
            '[a/Fe]': 0.2,
            'xi': 2.0,
            'RES': 28000,
            'abu_file': abu_file.name,
            'linelist_file': linelist_index.name,
            'line_file': line_file.name
        }
    ]
}
cfg_file.write_text(json.dumps(cfg, indent=2))

input_ts, cfg_file

## 4. Controlla i template creati

In [ ]:
print('--- input_tutorial.ts ---')
print(input_ts.read_text())
print('--- line_fit_config.tutorial.json ---')
print(cfg_file.read_text())

## 5. Comandi da eseguire

Esegui questi comandi in shell dalla root del repository.

1. Sintesi seriale
```bash
python main.py --input tutorial_data/input_tutorial.ts
```

2. Sintesi parallela
```bash
python main_parallel.py --input tutorial_data/input_tutorial.ts
```

3. Analisi line-by-line
```bash
python line_chi2_on_the_fly.py --config tutorial_data/line_fit_config.tutorial.json
```

## 6. Lettura rapida risultati (se presenti)

In [ ]:
import pandas as pd

best_files = sorted(results_dir.glob('*_best.csv'))
grid_files = sorted(results_dir.glob('*_grid.csv'))
print('best files:', [p.name for p in best_files])
print('grid files:', [p.name for p in grid_files])

if best_files:
    df_best = pd.read_csv(best_files[0])
    display(df_best.head())

if grid_files:
    df_grid = pd.read_csv(grid_files[0])
    display(df_grid.head())

## 7. Prossimi passi consigliati

- sostituisci `dummy_linelist_file.lin` con linelist reali;
- usa uno spettro osservato reale (colonne: lambda, flux, err opzionale, telluric opzionale);
- regola griglia `xfe_min/xfe_max/xfe_step` in funzione del segnale delle linee;
- attiva `mcmc=true` nel JSON quando vuoi stimare errori Monte Carlo.